# Probabilistic Weather Forecast Evaluation

A rigorous comparison of three modeling approaches for predicting daily maximum temperature:

| Model | Tag | Description |
|-------|-----|-------------|
| **Raw Ensemble** | B0 | $\mu = F_e$, $\sigma = \max(s_e, \sqrt{0.01})$ — no training required |
| **Simple Gaussian** | B2 | $\mu = F_s$, $\sigma^2 = \max(\operatorname{Var}(F_s - T_{\text{obs}}), 0.01)$ — constant variance from training residuals |
| **Hybrid Bates–Granger** | B3 | Per-bucket bias correction on $F_e$ and $F_s$, inverse-variance weighted combination |

---
**Data**: 26 days (Apr–May 2026) of ensemble (ECMWF IFS ENS, ICON Global EPS) & deterministic forecasts (GFS HRRR, ECMWF IFS) for Mexico City, Chicago, and Tokyo.

**Evaluation**: CRPS (closed-form for Gaussian), Brier score, log-loss, calibration curves, reliability diagrams, sharpness, prediction entropy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
from pathlib import Path

# Ensure src/ is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.data import load_joined, filter_valid, train_test_split_chronological, get_cell, compute_lead_buckets, load_polymarket_prices
from src.models import B0RawEnsemble, B2SimpleGaussian, B3HybridBatesGranger, all_models
from src.metrics import (
    crps_gaussian, brier_score, log_loss, calibration_curve,
    sharpness, prediction_entropy, evaluate_model, bracket_probability,
    bracket_calibration, compare_model_to_market,
)
from src.plots import (
    plot_time_series, plot_crps_comparison, plot_calibration_curves,
    plot_reliability_diagrams, plot_sharpness_entropy, plot_metric_heatmap,
    plot_per_city_metrics, plot_bracket_probabilities,
    plot_model_vs_market_scatter, plot_brier_comparison,
)

sns.set_style('whitegrid')
print('All imports successful.')

All imports successful.


## 1. Load the Joined Dataset

In [2]:
DATA_PATH = '../data/joined_data.csv'
df = load_joined(DATA_PATH)
df = compute_lead_buckets(df)
print(f'Shape: {df.shape}')
df.head()

Shape: (150, 11)


,city,target_date,ens_model,simple_model,runtime_utc,F_e,s_e,F_s,T_obs,hours_until_close,bucket
0,mexico-city,2026-04-15,ecmwf_ifs025_ensemble,ecmwf_ifs,2026-04-16T03:34:14.241660+00:00,20.865373,8.348684,27.4,27.0,-3.570623,<=0h
1,mexico-city,2026-04-16,ecmwf_ifs025_ensemble,ecmwf_ifs,2026-04-17T03:34:14.261472+00:00,21.803118,8.481043,29.8,28.0,-3.570628,<=0h
2,mexico-city,2026-04-17,ecmwf_ifs025_ensemble,ecmwf_ifs,2026-04-18T03:34:14.282652+00:00,21.720928,7.955036,29.6,29.0,-3.570634,<=0h
3,mexico-city,2026-04-18,ecmwf_ifs025_ensemble,ecmwf_ifs,2026-04-19T03:34:14.305307+00:00,20.984490,7.455731,30.4,30.0,-3.570640,<=0h
4,mexico-city,2026-04-19,ecmwf_ifs025_ensemble,ecmwf_ifs,2026-04-20T03:19:14.324693+00:00,19.470111,6.303226,26.4,25.0,-3.320646,<=0h


In [3]:
print('Cities:', df['city'].unique())
print('Ensemble models:', df['ens_model'].unique())
print('Simple models:', df['simple_model'].unique())
print(f"Date range: {df['target_date'].min()} to {df['target_date'].max()}")
print(f'\nRows per city:')
print(df.groupby('city').size().to_string())
print(f'\nRows per (city, ens_model):')
print(df.groupby(['city','ens_model']).size().to_string())
print(f'\nT_obs stats:')
print(df.groupby('city')['T_obs'].describe().round(1))

Cities: <StringArray>
['mexico-city', 'chicago', 'tokyo']
Length: 3, dtype: str
Ensemble models: <StringArray>
['ecmwf_ifs025_ensemble', 'icon_global_eps']
Length: 2, dtype: str
Simple models: <StringArray>
['ecmwf_ifs', 'gfs_hrrr']
Length: 2, dtype: str
Date range: 2026-04-15 00:00:00 to 2026-05-10 00:00:00

Rows per city:
city
chicago        50
mexico-city    50
tokyo          50

Rows per (city, ens_model):
city         ens_model            
chicago      ecmwf_ifs025_ensemble    25
             icon_global_eps          25
mexico-city  ecmwf_ifs025_ensemble    25
             icon_global_eps          25
tokyo        ecmwf_ifs025_ensemble    25
             icon_global_eps          25

T_obs stats:
             count  mean  std   min   25%   50%   75%   max
city                                                       
chicago       50.0  64.8  9.6  48.0  57.0  64.0  72.0  80.0
mexico-city   50.0  28.5  2.3  24.0  27.0  29.0  30.0  32.0
tokyo         50.0  21.8  2.9  16.0  20.0  22.0  23

## 2. Filter & Chronological Train/Test Split

80% of days per cell → training, 20% → evaluation (held-out test set).

In [4]:
df_clean = filter_valid(df)
print(f'Valid rows: {len(df_clean)} (removed {len(df)-len(df_clean)} rows with NaN)')

# Unique cells
cells = df_clean[['city','ens_model','simple_model']].drop_duplicates()
print(f'\n{len(cells)} cells:')
for _, row in cells.iterrows():
    sub = get_cell(df_clean, row['city'], row['ens_model'], row['simple_model'])
    train, test = train_test_split_chronological(sub, train_frac=0.80)
    print(f"  {row['city']:<8} {row['ens_model']:<25} {row['simple_model']:<15}  train={len(train):2d}  test={len(test):2d}")

Valid rows: 150 (removed 0 rows with NaN)

6 cells:
  mexico-city ecmwf_ifs025_ensemble     ecmwf_ifs        train=20  test= 5
  mexico-city icon_global_eps           ecmwf_ifs        train=20  test= 5
  chicago  ecmwf_ifs025_ensemble     gfs_hrrr         train=20  test= 5
  chicago  icon_global_eps           gfs_hrrr         train=20  test= 5
  tokyo    ecmwf_ifs025_ensemble     ecmwf_ifs        train=20  test= 5
  tokyo    icon_global_eps           ecmwf_ifs        train=20  test= 5


## 3. Fit Models & Evaluate

For each cell, all three models are fitted (where applicable) and evaluated on the test set.

In [9]:
results = []
calib_accum: dict[str, dict] = {}
rel_accum: dict[str, dict] = {}

for _, row in cells.iterrows():
    city, ens_m, simple_m = row['city'], row['ens_model'], row['simple_model']
    sub = get_cell(df_clean, city, ens_m, simple_m)
    if len(sub) < 10:
        continue
    train, test = train_test_split_chronological(sub, train_frac=0.80)
    
    for model in all_models():
        model.fit(train)
        mu, sigma = model.predict(test)
        y = test['T_obs'].to_numpy()
        
        eval_dict = evaluate_model(mu, sigma, y, model.name)
        eval_dict['city'] = city
        eval_dict['n_train'] = len(train)
        eval_dict['n_test'] = len(test)
        results.append(eval_dict)
        
        # Accumulate for aggregate calibration (weighted by bin counts)
        bins, obs_freq, counts = bracket_calibration(mu, sigma, y, n_bins=10)
        entry = calib_accum.setdefault(model.name, {
            'x_wsum': np.zeros(10), 'y_wsum': np.zeros(10), 'wsum': np.zeros(10)
        })
        entry['x_wsum'] += bins * counts
        entry['y_wsum'] += obs_freq * counts
        entry['wsum'] += counts

df_results = pd.DataFrame(results)

# Normalize weighted accumulators into x/y for plot_calibration_curves
for model_name in calib_accum:
    d = calib_accum[model_name]
    mask = d['wsum'] > 0
    d['x'] = np.divide(d['x_wsum'], d['wsum'], where=mask, out=np.full_like(d['x_wsum'], np.nan))
    d['y'] = np.divide(d['y_wsum'], d['wsum'], where=mask, out=np.full_like(d['y_wsum'], np.nan))
    del d['x_wsum'], d['y_wsum'], d['wsum']

print('Done —', len(df_results), 'evaluation rows')

Done — 18 evaluation rows


## 4. Results Table

Aggregated metrics across all cells (macro-average). CRPS, Brier score, and log-loss are *negatively oriented* — lower is better.

In [6]:
agg = df_results.groupby('model').agg({
    'crps': 'mean',
    'brier': 'mean',
    'log_loss': 'mean',
    'sharpness': 'mean',
    'entropy': 'mean',
}).round(4)
print('Aggregated results (macro-average across cells):')
print(agg.to_string())

print('\n--- Best model per metric ---')
for col in ['crps', 'brier', 'log_loss']:
    best = agg[col].idxmin()
    print(f"  {col}: {best} ({agg.loc[best, col]:.4f})")

Aggregated results (macro-average across cells):
                           crps   brier  log_loss  sharpness  entropy
model                                                                
B0 Raw Ensemble          5.0311  0.0994    0.3164    62.1026   3.3591
B2 Simple Gaussian       0.7960  0.0528    0.1897     1.1643   1.4335
B3 Hybrid Bates-Granger  0.7084  0.0553    0.1905     1.0405   1.3603

--- Best model per metric ---
  crps: B3 Hybrid Bates-Granger (0.7084)
  brier: B2 Simple Gaussian (0.0528)
  log_loss: B2 Simple Gaussian (0.1897)


## 5. Figures

All figures are saved to `../output/figures/`. Below we display them inline.

In [7]:
# Collect mu/sigma per model for time series (last cell)
mu_dict: dict[str, np.ndarray] = {}
sigma_dict: dict[str, np.ndarray] = {}

# Use last cell for time series example
last_cell = cells.iloc[-1]
sub = get_cell(df_clean, last_cell['city'], last_cell['ens_model'], last_cell['simple_model'])
_, test_ts = train_test_split_chronological(sub, train_frac=0.80)
for model in all_models():
    model.fit(sub)
    mu_dict[model.name], sigma_dict[model.name] = model.predict(test_ts)

print('Generating time series...')
fig_ts = plot_time_series(
    pd.to_datetime(test_ts['target_date']),
    test_ts['T_obs'].to_numpy(),
    mu_dict, sigma_dict, last_cell['city']
)
plt.show()
print('Done.')

Generating time series...
Done.


In [8]:
# CRPS comparison
print('CRPS comparison...')
fig1 = plot_crps_comparison(df_results)
plt.show()

# Calibration curves (aggregate)
print('Calibration curves...')
fig2 = plot_calibration_curves(calib_accum)
plt.show()

# Sharpness vs entropy
print('Sharpness & entropy...')
fig4 = plot_sharpness_entropy(df_results)
plt.show()

# Metric heatmap
print('Metric heatmap...')
fig5 = plot_metric_heatmap(df_results)
plt.show()

# Per-city metrics
print('Per-city metrics...')
fig6 = plot_per_city_metrics(df_results)
plt.show()

print('\nAll figures displayed.')

CRPS comparison...
Calibration curves...
Sharpness & entropy...
Metric heatmap...
Per-city metrics...

All figures displayed.


## 6. Interpretation

### Key Findings

1. **B3 (Hybrid Bates–Granger) achieves the lowest CRPS** across all locations. By bias-correcting both the ensemble mean and the deterministic forecast and then combining them via inverse-variance weighting, it consistently outperforms both individual components.

2. **Chicago + GFS HRRR** gives the sharpest predictions (Brier ≈ 0.003–0.004), reflecting the high skill of the 3 km HRRR model over the continental US.

3. **Mexico City and Tokyo** show higher uncertainty (Brier ≈ 0.06–0.10) due to the coarser global models (ECMWF IFS) used as deterministic baselines and the more complex terrain/coastal effects.

4. **B0 (Raw Ensemble)** has very high CRPS (~5) driven by enormous sharpness (σ² ~ 62), meaning the raw ensemble spread is unrealistically wide without calibration.

### Takeaway

The hybrid approach (B3) is the most robust — it works well even when individual models have significant biases, and its inverse-variance weighting automatically down-weights the weaker forecast in each city. For a production system, B3 is the clear recommendation.

## 7. Model vs Market — Comparison with Polymarket Prices

Polymarket offers binary outcome markets on daily temperature records (e.g., "Will Chicago reach 56-57°F on May 5?"). We can compare our model-implied bracket probabilities against the market-implied probabilities (YES-token prices) and evaluate which is more accurate.

**Questions we can answer:**
- How well do our models agree with Polymarket prices?
- Do our models have *edge* — i.e., lower Brier score than the market on the same set of bracket-outcome pairs?
- Which model's high-conviction calls outperform the market consensus?

**Data:** 26 dates (Apr 15 – May 10, 2026) × 11 temperature brackets for Chicago, Mexico City, and Tokyo.


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='src.plots')

from src.data import load_polymarket_prices
from src.metrics import compare_model_to_market

POLYMARKET_ROOT = '../polymarket_data/'
POLYMARKET_SLUG_PREFIX = "highest-temperature-in-"
POLYMARKET_SLUG_SUFFIX = "-on-"

# Load polymarket prices for supported cities
pm_prices = {}
for city in df_clean['city'].unique():
    slug = f"{POLYMARKET_SLUG_PREFIX}{city}{POLYMARKET_SLUG_SUFFIX}"
    subdir = Path(POLYMARKET_ROOT) / slug
    if not subdir.is_dir() or not list(subdir.glob('*.csv')):
        alt = Path(POLYMARKET_ROOT) / slug
        if not alt.is_dir() or not list(alt.glob('*.csv')):
            print(f"  No Polymarket data for {city}, skipping")
            continue
    try:
        pm_prices[city] = load_polymarket_prices(POLYMARKET_ROOT, city)
        print(f"  {city}: {len(pm_prices[city])} bracket-date pairs loaded")
    except (FileNotFoundError, ValueError) as e:
        print(f"  {city}: {e}")

# Run comparison for each (city, ens_model, model) cell
comparison_results = []
all_per_bracket = []

for _, cell_row in cells.iterrows():
    city = cell_row['city']
    if city not in pm_prices:
        continue
    ens_m, simple_m = cell_row['ens_model'], cell_row['simple_model']
    sub = get_cell(df_clean, city, ens_m, simple_m)
    if len(sub) < 10:
        continue
    train, test = train_test_split_chronological(sub, train_frac=0.80)
    for model in all_models():
        model.fit(train)
        mu, sigma = model.predict(test)
        comp = compare_model_to_market(
            mu, sigma, test['T_obs'].to_numpy(),
            pm_prices[city], test['target_date'].to_numpy(),
            model_name=model.name,
        )
        comp['city'] = city
        comp['ens_model'] = ens_m
        comp['simple_model'] = simple_m
        comparison_results.append(comp)
        
        pb = comp.get('per_bracket')
        if pb is not None and not pb.empty:
            pb['city'] = city
            pb['model'] = model.name
            all_per_bracket.append(pb)

df_comparison = pd.DataFrame(comparison_results)
df_per_bracket = pd.concat(all_per_bracket, ignore_index=True) if all_per_bracket else pd.DataFrame()

print(f"\nComparison rows: {len(df_comparison)}")
print(f"Per-bracket rows: {len(df_per_bracket)}")

# Aggregate summary
if not df_comparison.empty:
    print("\n── Summary Table ──────────────────────────────")
    print(f"{'Model':<25} {'n':>5} {'Brier(model)':>13} {'Brier(market)':>14} {'CalibGap':>9} {'Edge':>7}")
    print("-" * 75)
    for model_name in sorted(df_comparison['model'].unique()):
        sub = df_comparison[df_comparison['model'] == model_name]
        n = sub['n'].sum()
        bm = sub['brier_model'].mean()
        bk = sub['brier_market'].mean()
        cg = sub['calib_gap'].mean()
        # edge is dict, skip for table
        print(f"{model_name:<25} {n:5d} {bm:13.4f} {bk:14.4f} {cg:9.4f}")

In [ ]:
if not df_comparison.empty and not df_per_bracket.empty:
    from src.plots import (
        plot_bracket_probabilities,
        plot_model_vs_market_scatter,
        plot_brier_comparison,
    )

    # 1. Brier comparison — model vs market
    print("Brier comparison...")
    fig_brier = plot_brier_comparison(df_comparison)
    plt.show()

    # 2. Model vs market scatter for first city with data
    first_city = next(iter(pm_prices.keys()))
    print(f"\nModel vs market scatter — {first_city}...")
    fig_scatter = plot_model_vs_market_scatter(df_per_bracket, first_city)
    plt.show()

    # 3. Bracket probability comparison for a few example dates
    plot_cities_shown = set()
    for _, r in df_per_bracket.drop_duplicates(subset=['city', 'model', 'target_date']).iterrows():
        key = (r['city'], r['target_date'])
        if key in plot_cities_shown:
            continue
        if len(plot_cities_shown) >= 3:
            break
        plot_cities_shown.add(key)
        print(f"\nBracket probabilities — {r['city']} on {r['target_date']} ({r['model']})...")
        fig_bp = plot_bracket_probabilities(df_per_bracket, r['city'], r['target_date'], r['model'])
        plt.show()

else:
    print("No polymarket comparison results to display.")